# 🏗️ Day 4A — Schema Design: Building TaxDocument
**Agentic AI for Tax Technologists · 12-Day Program**

---

## 📖 The Story So Far

| Day | Built | Gap left open |
|-----|-------|---------------|
| Day 2 | ReAct agent — tool calls + short-term memory | Reasoning was implicit |
| Day 3 | CoT + few-shot + `TaxClassification` Pydantic schema | Inputs were clean typed questions |
| **Day 4A** | **Extend schema for unstructured real-world documents** | — |

**The Day 3 `TaxClassification` assumed a clean input:** someone typed a well-formed question.  
**Day 4 reality:** the input is a client email with typos, an invoice with a missing period, or a letter written by a human for a human.

The same Pydantic pattern applies — but the schema needs to be **designed for failure modes**, not just the happy path.

---

## 🎯 Learning Objectives
1. Understand what makes a schema fragile vs robust for document extraction
2. Design a `TaxDocument` Pydantic model with Optional fields and graceful degradation
3. Add a `missing_fields` tracker so downstream workflows know what to chase
4. Write and test field validators — currency normaliser, period parser
5. Understand the difference between `extraction_confidence` and Day 3's `confidence`

---
## ⏱️ Time: 30 minutes

---
## ⚙️ Step 1: Install & Import

In [ ]:
pip install openai pydantic --quiet

In [ ]:
import os
import re
import json
import datetime
from typing import Literal, Optional
from openai import AzureOpenAI
from getpass import getpass
from pydantic import BaseModel, Field, field_validator, model_validator, ValidationError

print("✅ Imports OK")

---
## 🔑 Step 2: Configure Azure OpenAI Client

In [ ]:
AZURE_OPENAI_ENDPOINT = input("Endpoint (e.g. https://xxx.openai.azure.com/): ").strip()
AZURE_OPENAI_API_KEY  = getpass("API Key: ")
AZURE_DEPLOYMENT_NAME = input("Deployment name (e.g. gpt-4o): ").strip()
AZURE_API_VERSION     = "2024-08-01-preview"

client = AzureOpenAI(
    azure_endpoint = AZURE_OPENAI_ENDPOINT,
    api_key        = AZURE_OPENAI_API_KEY,
    api_version    = AZURE_API_VERSION,
)

print("✅ Azure OpenAI client initialised successfully!")

---
## 📚 Step 3: The Problem — Why Day 3's Schema Breaks on Real Documents

Look at these two inputs side by side.

In [ ]:
# Day 3 input — clean, structured, written for a machine
DAY3_INPUT = "A Mumbai-based software company sells a Rs 2 lakh SaaS license to a registered client in Hyderabad."

# Day 4 input — real client email, written for a human
DAY4_INPUT = """
From: rajesh.kumar@alphacorp.in
Subject: GST query - urgent

Hi,

We need clarity on the GST for last quarter. Alpha Corp Pvt Ltd has done some
consulting work for a client in Dubai — roughly around 12 lakhs or so (haven't
finalised the invoice yet). Also some local work in Karnataka, maybe 8 lakhs?
Tax period should be Q3 FY25 but could be Q2 also, not 100% sure.

Please advise on what we need to file.

Thanks
Rajesh
"""

print("Day 3 input (clean):")
print(f"  {DAY3_INPUT}")
print()
print("Day 4 input (real email):")
print(DAY4_INPUT)

print("Fields a Day 3 schema would fail on:")
print("  ❌  liability_amount: float  →  'roughly 12 lakhs or so' is not a float")
print("  ❌  tax_period: str          →  'Q3 or Q2 FY25' is ambiguous")
print("  ❌  jurisdiction: Literal[]  →  'Dubai + Karnataka' is two jurisdictions")
print("  ❌  confidence: float        →  Day 3 confidence = answer certainty, not extraction certainty")

---
## 🧱 Step 4: Build the TaxDocument Schema

Design principles we apply:
1. `Optional` for anything a document might omit
2. `missing_fields: list[str]` to track what we could not extract
3. `Literal` for high-stakes controlled vocabulary fields
4. `extraction_confidence` — how sure are we the extraction is correct (not the answer)
5. Field validators to normalise messy values (e.g. 'rupees' → 'INR')

In [ ]:
class TaxDocument(BaseModel):
    """
    Structured extraction target for an unstructured tax-related document.
    Designed for real-world messiness: optional fields, missing-field tracking,
    and extraction-level confidence (distinct from answer confidence).
    """

    # ── Core identity fields ───────────────────────────────────────────────
    entity_name: Optional[str] = Field(
        default=None,
        description="Legal entity name (company or individual). None if not stated."
    )
    tax_period: Optional[str] = Field(
        default=None,
        description="Tax period, e.g. 'Q3 FY2025', 'FY2024-25', 'October 2024'. None if ambiguous or missing."
    )

    # ── Transaction fields ─────────────────────────────────────────────────
    transaction_type: Optional[str] = Field(
        default=None,
        description="Type of transaction: e.g. 'B2B Supply', 'Export of Services', 'Import RCM', 'Mixed'."
    )
    jurisdiction: Optional[str] = Field(
        default=None,
        description="State(s) or country/countries involved. Free text — can be 'Karnataka + Export'."
    )
    liability_amount: Optional[float] = Field(
        default=None,
        description="Tax liability or invoice amount in the stated currency. None if only approximate or missing."
    )
    currency: Literal["INR", "USD", "EUR", "GBP", "AED", "SGD", "Unknown"] = Field(
        default="INR",
        description="Currency of the liability amount. Default INR if not stated."
    )
    applicable_section: Optional[str] = Field(
        default=None,
        description="GST/income tax section if identifiable from document. None if not mentioned."
    )

    # ── Extraction quality fields ──────────────────────────────────────────
    extraction_confidence: float = Field(
        ge=0.0, le=1.0,
        description=(
            "How confident are we that the extracted fields are CORRECT? "
            "Distinct from Day 3 confidence (which was about answer accuracy). "
            "Low if document is ambiguous, approximate, or contradictory."
        )
    )
    missing_fields: list[str] = Field(
        default_factory=list,
        description="List of field names that could not be extracted from this document."
    )
    extraction_notes: Optional[str] = Field(
        default=None,
        description="Free-text notes explaining ambiguities or assumptions made during extraction."
    )
    human_review_required: bool = Field(
        default=False,
        description="True if extraction_confidence < 0.75 or any critical fields are missing."
    )

    # ── Validators ────────────────────────────────────────────────────────
    @field_validator("currency", mode="before")
    @classmethod
    def normalise_currency(cls, v):
        """Normalise messy currency strings to canonical Literal values."""
        if v is None:
            return "INR"
        mapping = {
            "rupee": "INR", "rupees": "INR", "rs": "INR", "inr": "INR", "₹": "INR",
            "dollar": "USD", "dollars": "USD", "usd": "USD", "$": "USD",
            "euro": "EUR", "eur": "EUR", "€": "EUR",
            "pound": "GBP", "gbp": "GBP", "£": "GBP",
            "dirham": "AED", "aed": "AED",
            "sgd": "SGD", "singapore dollar": "SGD",
        }
        return mapping.get(str(v).lower().strip(), "Unknown")

    @field_validator("liability_amount", mode="before")
    @classmethod
    def parse_amount(cls, v):
        """Parse amounts like '12 lakhs', '₹5L', '5,00,000' to float."""
        if v is None:
            return None
        if isinstance(v, (int, float)):
            return float(v)
        s = str(v).lower().replace(",", "").replace("₹", "").replace("rs", "").strip()
        # Handle lakh notation
        lakh_match = re.search(r"([\d.]+)\s*l(?:akh)?", s)
        if lakh_match:
            return float(lakh_match.group(1)) * 100000
        # Handle crore notation
        crore_match = re.search(r"([\d.]+)\s*cr(?:ore)?", s)
        if crore_match:
            return float(crore_match.group(1)) * 10000000
        # Plain number
        num_match = re.search(r"[\d.]+", s)
        if num_match:
            return float(num_match.group())
        return None

    @model_validator(mode="after")
    def auto_set_review_flag(self):
        """Auto-raise human_review_required if confidence is low or critical fields missing."""
        critical = {"entity_name", "tax_period", "transaction_type"}
        missing_critical = critical.intersection(set(self.missing_fields))
        if self.extraction_confidence < 0.75 or missing_critical:
            self.human_review_required = True
        return self


print("✅ TaxDocument schema defined")
print("\nFields:")
for fname, finfo in TaxDocument.model_fields.items():
    req = "required" if finfo.is_required() else f"default={finfo.default!r}"
    print(f"  {fname:<28} [{req}]")

---
## 🧪 Step 5: Test the Validators Directly

In [ ]:
# Test currency normaliser
print("CURRENCY NORMALISER TESTS")
print("-" * 40)
test_currencies = ["rupees", "Rs", "₹", "$", "dollar", "USD", "EUR", "dirham", "bitcoin", None]
for c in test_currencies:
    result = TaxDocument(
        extraction_confidence=0.9,
        currency=c,
    ).currency
    print(f"  {str(c):<12} → {result}")

print()
print("AMOUNT PARSER TESTS")
print("-" * 40)
test_amounts = ["12 lakhs", "₹5L", "2.5 crore", "5,00,000", "Rs 8 lakh", "approximately 12L", None, 150000]
for a in test_amounts:
    result = TaxDocument(
        extraction_confidence=0.9,
        liability_amount=a,
    ).liability_amount
    formatted = f"₹{result:,.0f}" if result is not None else "None"
    print(f"  {str(a):<25} → {formatted}")

---
## 🔬 Step 6: Instantiate with Realistic Partial Data

In [ ]:
# Simulate what an extractor would return for the Rajesh email
partial_extraction = TaxDocument(
    entity_name          = "Alpha Corp Pvt Ltd",
    tax_period           = None,             # ambiguous — Q2 or Q3?
    transaction_type     = "Mixed — Export of Services + B2B Intra-State",
    jurisdiction         = "Karnataka + Dubai (Export)",
    liability_amount     = None,             # 'roughly 12 lakhs' is not a reliable figure
    currency             = "INR",
    applicable_section   = None,
    extraction_confidence = 0.52,
    missing_fields       = ["tax_period", "liability_amount"],
    extraction_notes     = (
        "Amount stated as 'roughly 12 lakhs or so' — not extracted as figure. "
        "Tax period ambiguous: sender said 'Q3 FY25 but could be Q2'."
    ),
)

print("Partial extraction result:")
print("-" * 55)
for field, value in partial_extraction.model_dump().items():
    flag = " ← MISSING" if field == "missing_fields" and value else ""
    flag = " ← AUTO-FLAGGED" if field == "human_review_required" and value else flag
    print(f"  {field:<28} : {value}{flag}")

---
## 📐 Step 7: Build the JSON Schema Dict for Function Calling

In Notebook 4B we pass this schema to Azure OpenAI as a `tools` parameter.  
The model is then constrained to return ONLY data matching this schema — no fences, no prose.

In [ ]:
# Build the OpenAI-compatible function schema from our Pydantic model
EXTRACTION_FUNCTION_SCHEMA = {
    "type": "function",
    "function": {
        "name": "extract_tax_document",
        "description": (
            "Extract structured tax compliance fields from an unstructured document "
            "(email, letter, invoice snippet). Return partial results for missing fields — "
            "do not hallucinate values."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "entity_name": {
                    "type": ["string", "null"],
                    "description": "Legal entity name. Null if not stated."
                },
                "tax_period": {
                    "type": ["string", "null"],
                    "description": "Tax period (e.g. Q3 FY2025, October 2024). Null if ambiguous."
                },
                "transaction_type": {
                    "type": ["string", "null"],
                    "description": "Transaction type: B2B Supply, Export of Services, Import RCM, Mixed, etc."
                },
                "jurisdiction": {
                    "type": ["string", "null"],
                    "description": "State(s) or countries involved."
                },
                "liability_amount": {
                    "type": ["number", "null"],
                    "description": "Numeric amount in rupees (or stated currency). Null if approximate or missing."
                },
                "currency": {
                    "type": "string",
                    "enum": ["INR", "USD", "EUR", "GBP", "AED", "SGD", "Unknown"],
                    "description": "Currency. Default INR."
                },
                "applicable_section": {
                    "type": ["string", "null"],
                    "description": "GST/tax section if identifiable from document."
                },
                "extraction_confidence": {
                    "type": "number",
                    "minimum": 0.0,
                    "maximum": 1.0,
                    "description": "Confidence that extracted fields are correct. Low if document is ambiguous."
                },
                "missing_fields": {
                    "type": "array",
                    "items": {"type": "string"},
                    "description": "Field names that could not be extracted."
                },
                "extraction_notes": {
                    "type": ["string", "null"],
                    "description": "Notes on ambiguities or assumptions."
                },
            },
            "required": [
                "extraction_confidence",
                "missing_fields",
                "currency",
            ]
        }
    }
}

print("✅ EXTRACTION_FUNCTION_SCHEMA built")
print(f"   Properties defined: {len(EXTRACTION_FUNCTION_SCHEMA['function']['parameters']['properties'])}")
print(f"   Required fields:    {EXTRACTION_FUNCTION_SCHEMA['function']['parameters']['required']}")

# Save schema for import in Notebooks 4B and 4C
with open("day4_schema.json", "w") as f:
    json.dump(EXTRACTION_FUNCTION_SCHEMA, f, indent=2)
print("\n✅ Schema saved to day4_schema.json — import this in 4B and 4C")

---
## 🎓 Key Insight

```
Day 3 TaxClassification   ← clean input  → every field always present
Day 4 TaxDocument         ← messy input  → Optional fields, missing tracker, partial OK
```

The key new design decisions:
- `Optional[float]` on `liability_amount` — never crash on "roughly 12 lakhs"
- `list[str]` on `missing_fields` — tell downstream what to chase, don't silently swallow None
- `extraction_confidence` ≠ Day 3 `confidence` — different question entirely
- `model_validator` auto-raises review flag on critical missing fields

---
## ➡️ Next: Notebook 4B — Extraction Engine

We plug this schema into Azure OpenAI's function calling API and extract from 5 real-world documents — a clean email, an ambiguous letter, an invoice snippet, a multi-entity document, and a near-empty message.

# Setup Local

1. Create a Folder named “agentic-ai-ttt”
2. Open the folder in VS Code
3. Virtual Environment
4. Command to active:
venv\Source\activate